# Appendix A - Numerical Methods Behind SciBmad

This appendix collects several numerical ideas that appear throughout the tutorial, often without being named explicitly every time. The goal is not to replace a numerical analysis textbook, but to explain the computational backbone behind common SciBmad workflows: tracking particles, building maps, finding closed orbits, correcting errors, and fitting lattice parameters.

A useful way to read the main chapters is to keep the following hierarchy in mind:

$$
\text{lattice elements}
\quad \longrightarrow \quad
\text{tracking methods}
\quad \longrightarrow \quad
\text{maps and response matrices}
\quad \longrightarrow \quad
\text{analysis and solvers}.
$$

The high-level functions such as `track`, `twiss`, `find_closed_orbit`, `dynamic_aperture`, and the optimization routines all sit on top of this lower-level machinery.


## A.1 Tracking Methods

In SciBmad, a `tracking_method` specifies how a beamline element is traversed numerically. It is therefore more than a setting for long-term particle tracking. It defines the local map through an element, and that local map can later be used by many higher-level calculations.

For example, the same element-by-element propagation machinery can enter:

- particle tracking,
- closed-orbit finding,
- Twiss and normal-form calculations,
- dynamic aperture studies,
- radiation-damping studies,
- response-matrix construction,
- optimization and error fitting.

A tracking method may control the integration order, the number of integration steps, the longitudinal step size, fringe-field treatment, and optional effects such as radiation damping or fluctuations. Thus a lattice is not only a list of magnets and field strengths; it is also a prescription for how the equations of motion are evaluated through those magnets.

If an element has length $L$, a tracking method usually replaces the exact element map

$$
z_{\mathrm{out}} = M_L(z_{\mathrm{in}})
$$

with a numerical composition of smaller maps,

$$
M_L \approx M_{\Delta s_N} \circ M_{\Delta s_{N-1}} \circ \cdots \circ M_{\Delta s_1}.
$$

Here $z$ denotes the phase-space coordinates, and the step maps $M_{\Delta s_i}$ are chosen so that the approximation is accurate and, when appropriate, symplectic.

A tracking method can be assigned explicitly when constructing a beamline element. For example,

```julia
qf = Quadrupole(
    L = 1.0,
    Kn1 = 0.5,
    tracking_method = Yoshida(order=4, n_steps=8),
)
```

It can also be changed later by assigning to the element field:

```julia
ele.tracking_method = Yoshida(order=4, n_steps=8)
```

This is useful when we want explicit control over the numerical integration used by a specific element or by all elements in a ring.

The default tracking method for a SciBmad `LineElement` is `SciBmadStandard()`. This default method uses exact transport maps when they are available, and otherwise dispatches to Yoshida-style symplectic integration for finite-length elements. Thus, although an element may display `tracking_method = SciBmadStandard()`, the underlying integration used in many practical cases is Yoshida, which we discuss in detail in the next section.

Besides `Yoshida`, SciBmad also provides specialized tracking methods such as `Exact`, `MatrixKick`, `BendKick`, `SolenoidKick`, `DriftKick`, and `SaganCavity`.


## A.2 Yoshida Symplectic Integration

The basic idea is to split the Hamiltonian into simpler pieces. In an idealized form, suppose

$$
H(q,p) = T(p) + V(q),
$$

where $T$ generates drift-like motion and $V$ generates kick-like motion. The exact one-step map is formally

$$
\exp\left(h L_H\right)
=
\exp\left(h(L_T + L_V)\right),
$$

where $L_H$ is the Lie operator associated with the Hamiltonian $H$. Since the two pieces are usually easier to apply separately, a second-order symmetric splitting is

$$
S_2(h)
=
\exp\left(\frac{h}{2}L_T\right)
\exp\left(hL_V\right)
\exp\left(\frac{h}{2}L_T\right).
$$

This is the familiar drift-kick-drift structure. Yoshida's construction builds higher-order symplectic integrators by composing lower-order ones with carefully chosen coefficients. A fourth-order Yoshida composition can be written as

$$
S_4(h) = S_2(w_1 h) S_2(w_0 h) S_2(w_1 h),
$$

with

$$
w_1 = \frac{1}{2 - 2^{1/3}},
\qquad
w_0 = -\frac{2^{1/3}}{2 - 2^{1/3}}.
$$

The coefficients cancel lower-order error terms while preserving the symplectic structure, because a composition of symplectic maps is still symplectic.

This matters for storage-ring calculations. A non-symplectic method can introduce artificial damping, artificial excitation, or slow phase-space distortion over many turns. A symplectic method does not make the integration exact, but it keeps the long-term geometry of the motion much closer to the Hamiltonian system being modeled.

In practical SciBmad usage, parameters such as `order`, `n_steps`, and `ds_step` control the tradeoff between speed and accuracy. A larger number of steps or a higher order generally improves the element map, but increases the cost of tracking. Radiation damping and stochastic radiation can also be inserted into the tracking process; when these effects are enabled, the dynamics is no longer purely Hamiltonian, but the integration framework still organizes where those effects are applied.

Therefore, when we want to control the Yoshida integrator directly, we can manually specify both the order and the step subdivision. The example in last section,

```julia
tracking_method = Yoshida(order=4, n_steps=8)
```

uses a fourth-order Yoshida integrator and divides the element into eight integration steps. Equivalently, one may specify a target step length with `ds_step` instead of specifying `n_steps`.


## A.3 Optimization Methods and `Optim.jl`

Many accelerator-design tasks can be written as scalar optimization problems. We choose a vector of adjustable variables

$$
u = (u_1, u_2, \ldots, u_n),
$$

where the entries might be quadrupole strengths, corrector kicks, sextupole strengths, RF parameters, or alignment errors. We then define a scalar objective function that measures how far the current lattice is from the desired design.

In several tutorial workflows, we use Julia's `Optim.jl` package as a general-purpose minimizer. The usual pattern is to define a merit function

$$
\chi^2(u) = \frac{1}{2}\|r(u)\|_2^2,
$$

and then let an optimization method choose trial updates for $u$:

```julia
objective(u) = 0.5 * sum(abs2, residuals(u))

result = optimize(objective, u0, method)
```

Conceptually, `Optim.jl` sits above the accelerator model. SciBmad evaluates the lattice and the objective, while `Optim.jl` decides how to change the variables.

### A.3.1 Derivative-free methods

If gradients are unavailable, unreliable, or expensive, a derivative-free method such as `NelderMead()` can be useful:

```julia
result = optimize(objective, u0, NelderMead())
```

`NelderMead()` is often convenient for low-dimensional exploratory matching problems. It does not require derivatives, but it usually becomes inefficient as the number of variables grows.

### A.3.2 Supplying gradients manually

Many `Optim.jl` methods can use gradients supplied by the user. This is common in the tutorial because we often know how to compute derivatives more efficiently or more accurately than by letting the optimizer guess them internally.

A typical in-place gradient function has the form

```julia
function gradient!(G, u)
    # Fill G with d(objective)/du evaluated at u.
    G .= computed_gradient(u)
    return G
end
```

Then the optimizer can be called with both the objective and the gradient:

```julia
result = optimize(objective, gradient!, u0, BFGS())
```

The gradient may come from an analytic expression, finite differences, automatic differentiation, GTPSA, or a response-matrix calculation. Supplying the gradient explicitly often makes convergence faster and more reliable, especially when the variables have different scales or when the objective is expensive to evaluate.

### A.3.3 Quasi-Newton methods

When gradients are available, quasi-Newton methods are usually more efficient. Common choices are `BFGS()` and `LBFGS()`:

`BFGS()` builds an approximation to the local Hessian of the objective. `LBFGS()` is a limited-memory version that is better suited to larger numbers of variables. In this tutorial, gradients may come from finite differences, automatic differentiation, or GTPSA.

### A.3.4 Bounded optimization

Accelerator variables often have physical limits. For example, magnet strengths or corrector kicks may be restricted to an allowed range. In that case, one can use a bounded optimizer such as `Fminbox` wrapped around a method like `LBFGS()`:

```julia
result = optimize(objective, lower, upper, u0, Fminbox(LBFGS()))
```

The same idea appears in L-BFGS-B style optimizers: the optimizer searches for a minimum while keeping the variables inside specified bounds.

### A.3.5 Choosing a method

The method is chosen according to what information is available and what constraints the problem has:

| Situation | Typical method |
|---|---|
| No reliable gradients, few variables | `NelderMead()` |
| Gradients can be supplied, moderate variables | `BFGS()` |
| Gradients can be supplied, many variables | `LBFGS()` |
| Variables have lower/upper limits | `Fminbox(LBFGS())` or L-BFGS-B |
| Explicit residuals and response matrix | Gauss-Newton or damped least squares |

The last row is important enough to separate from generic scalar optimization. When the problem has a clear residual vector and response matrix, it is often better to use the least-squares structure directly, as described in the next section.


## A.4 Optimization with Response Matrices

Many accelerator correction and fitting problems have a special least-squares structure. Instead of thinking only about the scalar objective

$$
\chi^2(u) = \frac{1}{2}\|r(u)\|_2^2,
$$

we keep the residual vector

$$
r(u) = (r_1(u), r_2(u), \ldots, r_m(u)).
$$

The residuals might be orbit errors, tune errors, Twiss mismatch, chromaticity errors, coupling terms, or measured-minus-modeled beam-position readings. The local sensitivity of the residuals to the variables is the Jacobian, or response matrix,

$$
J_{ij} = \frac{\partial r_i}{\partial u_j}.
$$

Near the current parameter vector $u$, we linearize the residuals:

$$
r(u + \Delta u) \approx r(u) + J\Delta u.
$$

The response matrix can be computed by finite differences, automatic differentiation, or map-based methods such as GTPSA, depending on the problem and the available differentiability of the model.

### A.4.1 Gauss-Newton

The Gauss-Newton method uses the linearized residuals directly. It chooses $\Delta u$ by minimizing

$$
\min_{\Delta u} \left\| r + J\Delta u \right\|_2^2.
$$

By finding the derivative, the corresponding normal equations are

$$
J^T J\Delta u = -J^T r.
$$

This method is powerful when the model is locally close to linear and the response matrix is well conditioned. It is a natural fit for orbit correction, matching, and error fitting because those problems are often expressed directly in terms of residuals and response matrices.

### A.4.2 Damped least squares

In accelerator applications, response matrices are often nearly singular or strongly correlated. Damped least squares modifies the Gauss-Newton objective by adding a penalty on the size of the correction. Instead of minimizing only the residual norm, we minimize

$$
\min_{\Delta u}
\left(
\left\| r + J\Delta u \right\|_2^2
+
\lambda \left\|\Delta u\right\|_2^2
\right).
$$

Taking the derivative with respect to $\Delta u$ and setting it to zero gives

$$
\left(J^T J + \lambda I\right)\Delta u = -J^T r,
$$

where $\lambda \ge 0$ is the damping parameter. Increasing $\lambda$ makes the step smaller and more stable; decreasing $\lambda$ makes the method closer to ordinary Gauss-Newton. This is useful when a correction would otherwise demand unrealistically large magnet changes or when the response matrix has weakly constrained directions.

A weighted version is also common. If $W$ weights the residuals, then the objective is

$$
\min_{\Delta u}
\left(
\left\| W(r + J\Delta u) \right\|_2^2
+
\lambda \left\|\Delta u\right\|_2^2
\right),
$$

and the damped normal equations become

$$
\left(J^T W^T W J + \lambda I\right)\Delta u = -J^T W^T W r.
$$

Thus the correction and fitting workflow is often

$$
\text{compute residuals}
\quad \longrightarrow \quad
\text{compute response matrix}
\quad \longrightarrow \quad
\text{solve a Gauss-Newton or DLS step}
\quad \longrightarrow \quad
\text{update lattice}.
$$

Compared with a general `Optim.jl` call, Gauss-Newton and damped least squares exploit the special least-squares structure of the problem. They are less black-box and often easier to interpret physically, because the update is computed directly from the response matrix.


## A.5 Newton Solver for Closed Orbits

Closed-orbit finding also uses a linearized correction step, but it is not exactly the same algorithm as Gauss-Newton. Gauss-Newton minimizes a least-squares objective; while Newton's method here solves a nonlinear root-finding problem.

For closed-orbit finding, let $M$ be the one-turn map of the ring. Here the unknown is a phase-space coordinate vector, which we write as $\mathbf{v}$. For example,

$$
\mathbf{v} = (x, p_x, y, p_y, z, p_z)^T.
$$

A closed orbit $\mathbf{v}_{\mathrm{co}}$ satisfies the vector equation

$$
M(\mathbf{v}_{\mathrm{co}}) = \mathbf{v}_{\mathrm{co}}.
$$

Equivalently, define

$$
\mathbf{F}(\mathbf{v}) = M(\mathbf{v}) - \mathbf{v}.
$$

Then the closed orbit is a root of the vector function $\mathbf{F}:\mathbb{R}^6 \to \mathbb{R}^6$:

$$
\mathbf{F}(\mathbf{v}) = \mathbf{0}.
$$

Newton's method solves this nonlinear vector equation iteratively. At the current guess $\mathbf{v}_k$, we linearize

$$
\mathbf{F}(\mathbf{v}_k + \Delta \mathbf{v})
\approx
\mathbf{F}(\mathbf{v}_k) + \frac{\partial \mathbf{F}}{\partial \mathbf{v}}\bigg|_{\mathbf{v}_k}\Delta \mathbf{v}.
$$

Setting this approximation to zero gives the Newton correction equation

$$
\frac{\partial \mathbf{F}}{\partial \mathbf{v}}\bigg|_{\mathbf{v}_k}\Delta \mathbf{v} = -\mathbf{F}(\mathbf{v}_k),
$$

followed by the update

$$
\mathbf{v}_{k+1} = \mathbf{v}_k + \Delta \mathbf{v}.
$$

Since $\mathbf{F}(\mathbf{v}) = M(\mathbf{v}) - \mathbf{v}$, the Jacobian matrix is

$$
\frac{\partial \mathbf{F}}{\partial \mathbf{v}}
=
\frac{\partial M}{\partial \mathbf{v}} - I.
$$

Thus closed-orbit finding sits directly on top of tracking and differentiation. Tracking provides the one-turn map $M$, while differentiation provides the linearized map $\partial M/\partial \mathbf{v}$. In practice, the derivative can come from automatic differentiation, finite differences, or GTPSA-based map construction.

This is why closed-orbit finding is sensitive to the underlying tracking method. If the element maps change because the integrator order, step size, radiation settings, or fringe-field model changes, then the one-turn map $M$ changes, and the closed orbit may change as well.


## A.6 Phase Trombones and Phase Advance

A phase trombone is an artificial map that changes the betatron phase advance while preserving the local optical functions as much as possible. In Chapter 12, trombones are used to adjust the phasing between sextupoles and the interaction point, while compensating the net tune shift elsewhere in the ring.

The underlying idea comes from the eigenstructure of a stable symplectic map. Let $M$ be a one-turn map, or the linear map through a section of the ring. For $M$ to be a symplectic matrix, its eigenvalues must come in reciprocal pairs:

$$
\lambda \quad \Longrightarrow \quad \frac{1}{\lambda}.
$$

For a real matrix, complex eigenvalues also come with their complex conjugates. Stable transverse motion corresponds to an elliptic fixed point: the eigenvalues lie on the unit circle, so

$$
|\lambda| = 1.
$$

In that case the reciprocal is the complex conjugate, and each stable mode has the pair

$$
\lambda = e^{i\phi},
\qquad
\lambda^{-1} = e^{-i\phi}.
$$

The angle $\phi$ is the phase advance of that mode. For a one-turn map, the corresponding tune is

$$
\nu = \frac{\phi}{2\pi}.
$$

Equivalently, in a real normalized-coordinate basis, the same motion is represented by a rotation block,

$$
R(\phi)
=
\begin{pmatrix}
\cos\phi & \sin\phi \\
-\sin\phi & \cos\phi
\end{pmatrix}.
$$

Therefore transfer matrices can be block-diagonalize into 2 x 2 rotation blocks. A local trombone map can therefore be built by transforming to normalized coordinates, applying a phase rotation, and transforming back:

$$
M_{\mathrm{trombone}}
=
A\,R(\Delta\phi_x, \Delta\phi_y)\,A^{-1},
$$

where $A$ is the local normalizing map and $R(\Delta\phi_x, \Delta\phi_y)$ applies independent rotations in the two transverse normal modes. This is the same idea as the implementation pattern

```julia
M = A * R * inv(A)
```

used by the normalizing-map trombone utilities.

An ideal trombone changes phase without changing the local beta and alpha functions. A real lattice cannot usually insert such a perfect artificial map directly, so the practical matching problem is to vary quadrupole strengths so that the desired phase advance is produced while selected Twiss functions, especially at the IP, remain fixed. If the trombone changes the total phase around the ring, another trombone or matching section may be used to compensate the net tune shift.


## A.7 Summary

The algorithms in this appendix play different roles:

- Tracking methods describe how particles and maps pass through elements.
- Yoshida integration provides a symplectic drift-kick framework for long-term Hamiltonian dynamics.
- `Optim.jl` methods provide general scalar minimization tools such as Nelder-Mead, BFGS, L-BFGS, and bounded variants.
- Response-matrix methods exploit least-squares structure through Gauss-Newton and damped least squares.
- Newton solvers find fixed points such as closed orbits by repeatedly linearizing the one-turn map.
- Phase trombones use normal-mode rotations to adjust phase advance while preserving local optics as much as possible.

Together, these methods form the numerical layer underneath much of the tutorial. The user-facing SciBmad commands are compact, but they are built on this chain of tracking, differentiation, map construction, and linearized solving.
